In [1]:
if (!require(readr)) install.packages("readr")
if (!require(dplyr)) install.packages("dplyr")
if (!require(lubridate)) install.packages("lubridate")
if (!require(tidyr)) install.packages("tidyr")
if (!require(tidyr)) install.packages("ggplot2")


Загрузка требуемого пакета: readr

Warning message:
"пакет 'readr' был собран под R версии 4.4.3"
Загрузка требуемого пакета: dplyr


Присоединяю пакет: 'dplyr'


Следующие объекты скрыты от 'package:stats':

    filter, lag


Следующие объекты скрыты от 'package:base':

    intersect, setdiff, setequal, union


Загрузка требуемого пакета: lubridate

Warning message:
"пакет 'lubridate' был собран под R версии 4.4.3"

Присоединяю пакет: 'lubridate'


Следующие объекты скрыты от 'package:base':

    date, intersect, setdiff, union


Загрузка требуемого пакета: tidyr



In [2]:
library(readr)
library(dplyr)
library(lubridate)
library(tidyr)
library(ggplot2)


 ## Data Loading and Overviewing

In [3]:
events_data <- read_csv("C:\\Users\\safon\\Documents\\DS_Project_STEPIC\\data\\event_data_train.csv")
submissions_data <- read_csv("C:\\Users\\safon\\Documents\\DS_Project_STEPIC\\data\\submissions_data_train.csv")

Rows: 3480703 Columns: 4
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): action
dbl (3): step_id, timestamp, user_id

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 509104 Columns: 4
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): submission_status
dbl (3): step_id, timestamp, user_id

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
cat("events_data size:", format(object.size(events_data), units = "MB"), "\n")
cat("submissions_data size:", format(object.size(submissions_data), units = "MB"), "\n")

events_data size: 106.2 Mb 
submissions_data size: 15.5 Mb 


In [5]:
head(events_data,20)

step_id,timestamp,action,user_id
<dbl>,<dbl>,<chr>,<dbl>
32815,1434340848,viewed,17632
32815,1434340848,passed,17632
32815,1434340848,discovered,17632
32811,1434340895,discovered,17632
32811,1434340895,viewed,17632
32811,1434340895,passed,17632
32929,1434340921,discovered,17632
32929,1434340921,viewed,17632
32929,1434340939,started_attempt,17632


In [6]:
events_data %>% sample_n(10)

step_id,timestamp,action,user_id
<dbl>,<dbl>,<chr>,<dbl>
32954,1470205370,passed,22129
31979,1458059990,discovered,25494
34027,1504805937,discovered,16990
32191,1517584882,viewed,19243
33874,1494932525,viewed,1189
32219,1501949391,started_attempt,19029
31957,1501328279,passed,332
33493,1450414119,viewed,25551
32075,1525087132,viewed,8722


In [7]:
unique(events_data$action)

[1] "viewed"          "passed"          "discovered"      "started_attempt"

In [8]:
dim(events_data)

[1] 3480703       4

In [9]:
events_data %>% count(action) %>% mutate(p = n / sum(n))

action,n,p
<chr>,<int>,<dbl>
discovered,689495,0.19809073
passed,650331,0.18683898
started_attempt,310047,0.08907597
viewed,1830830,0.52599432


In [10]:
glimpse(events_data)

Rows: 3,480,703
Columns: 4
$ step_id   <dbl> 32815, 32815, 32815, 32811, 32811, 32811, 32929, 32929, 3292…
$ timestamp <dbl> 1434340848, 1434340848, 1434340848, 1434340895, 1434340895, …
$ action    <chr> "viewed", "passed", "discovered", "discovered", "viewed", "p…
$ user_id   <dbl> 17632, 17632, 17632, 17632, 17632, 17632, 17632, 17632, 1763…


In [11]:
summary(events_data)

    step_id         timestamp            action             user_id     
 Min.   : 30456   Min.   :1.434e+09   Length:3480703     Min.   :    1  
 1st Qu.: 32179   1st Qu.:1.450e+09   Class :character   1st Qu.: 6642  
 Median : 32929   Median :1.484e+09   Mode  :character   Median :13208  
 Mean   : 34874   Mean   :1.480e+09                      Mean   :13315  
 3rd Qu.: 33481   3rd Qu.:1.508e+09                      3rd Qu.:20144  
 Max.   :158433   Max.   :1.527e+09                      Max.   :26798  

### Note on step_id structure:
Step identifiers on the Stepik platform are global and not sequential within a course.
Large gaps and high numeric values (e.g., 150k+) are normal.
Step IDs are treated as categorical identifiers without semantic meaning.

In [12]:
events_data %>%
    summarise(across(everything(),n_distinct))

step_id,timestamp,action,user_id
<int>,<int>,<int>,<int>
198,2196932,4,19234


In [13]:
colSums(is.na(events_data))

step_id timestamp    action   user_id 
        0         0         0         0

In [14]:
sum(duplicated(events_data))

[1] 2333

In [15]:
head(submissions_data)

step_id,timestamp,submission_status,user_id
<dbl>,<dbl>,<chr>,<dbl>
31971,1434349275,correct,15853
31972,1434348300,correct,15853
31972,1478852149,wrong,15853
31972,1478852164,correct,15853
31976,1434348123,wrong,15853
31976,1434348188,correct,15853


In [16]:
submissions_data %>% sample_n(10)

step_id,timestamp,submission_status,user_id
<dbl>,<dbl>,<chr>,<dbl>
33685,1501252060,correct,9929
31976,1516564129,wrong,3726
32206,1522176717,wrong,20557
32812,1470913483,correct,21027
31971,1521407778,correct,9160
32173,1514274354,correct,3531
32244,1439027008,wrong,22882
33418,1488743862,correct,15431
33998,1525174955,correct,14891


In [17]:
unique(submissions_data$submission_status)

[1] "correct" "wrong"

In [18]:
submissions_data %>% count(submission_status) %>% mutate(p = n/sum(n))

submission_status,n,p
<chr>,<int>,<dbl>
correct,222705,0.437445
wrong,286399,0.562555


In [19]:
dim(submissions_data)

[1] 509104      4

In [20]:
glimpse(submissions_data)

Rows: 509,104
Columns: 4
$ step_id           <dbl> 31971, 31972, 31972, 31972, 31976, 31976, 31976, 319…
$ timestamp         <dbl> 1434349275, 1434348300, 1478852149, 1478852164, 1434…
$ submission_status <chr> "correct", "correct", "wrong", "correct", "wrong", "…
$ user_id           <dbl> 15853, 15853, 15853, 15853, 15853, 15853, 15853, 158…


In [21]:
summary(submissions_data)

    step_id         timestamp         submission_status     user_id     
 Min.   : 31971   Min.   :1.434e+09   Length:509104      Min.   :    2  
 1st Qu.: 32031   1st Qu.:1.450e+09   Class :character   1st Qu.: 6431  
 Median : 32763   Median :1.484e+09   Mode  :character   Median :13082  
 Mean   : 33733   Mean   :1.481e+09                      Mean   :13199  
 3rd Qu.: 33481   3rd Qu.:1.508e+09                      3rd Qu.:19985  
 Max.   :120745   Max.   :1.527e+09                      Max.   :26798  

### Note on step_id:
Step IDs on Stepik are global and not sequential within a course. Large numeric values and gaps are normal. The same pattern appears in both events and submissions, so step_id should be treated purely as a categorical identifier.

In [22]:
submissions_data %>%
    summarise(across(everything(),n_distinct))

step_id,timestamp,submission_status,user_id
<int>,<int>,<int>,<int>
76,504343,2,9940


In [23]:
colSums(is.na(submissions_data))

step_id         timestamp submission_status           user_id 
                0                 0                 0                 0

In [24]:
sum(duplicated(submissions_data))

[1] 107

In [25]:
length(setdiff(unique(events_data$user_id), unique(submissions_data$user_id)))
length(setdiff(unique(events_data$step_id), unique(submissions_data$step_id)))

[1] 9294

[1] 122

In [26]:
length(setdiff(unique(submissions_data$user_id), unique(events_data$user_id)))
length(setdiff(unique(submissions_data$step_id), unique(events_data$step_id)))

[1] 0

[1] 0

## EDA on Raw Event-Level Data

In [27]:
events_data <- events_data[!duplicated(events_data),]
submissions_data <- submissions_data[!duplicated(submissions_data),]

In [28]:
sum(duplicated(events_data))
sum(duplicated(submissions_data))

[1] 0

[1] 0

In [29]:
events_data$timestamp_dt <- as_datetime(events_data$timestamp)
submissions_data$timestamp_dt <- as_datetime(submissions_data$timestamp)

In [30]:
summary(events_data$timestamp_dt)
summary(submissions_data$timestamp_dt)

                      Min.                    1st Qu. 
"2015-06-15 04:00:48.0000" "2015-12-16 06:23:43.2500" 
                    Median                       Mean 
"2017-01-08 18:23:00.0000" "2016-11-27 21:59:13.7051" 
                   3rd Qu.                       Max. 
"2017-10-13 10:38:29.5000" "2018-05-19 23:33:31.0000" 

                      Min.                    1st Qu. 
"2015-06-15 04:02:24.0000" "2015-12-13 00:52:11.0000" 
                    Median                       Mean 
"2017-01-13 21:33:35.0000" "2016-12-01 10:27:40.5671" 
                   3rd Qu.                       Max. 
"2017-10-18 12:12:03.0000" "2018-05-19 23:07:07.0000" 

In [31]:
passed_counts <- events_data %>%
  filter(action == "passed") %>%
  group_by(user_id, step_id) %>%
  summarise(passed_count = n(), .groups = "drop")

multiple_passed <- passed_counts %>%
  filter(passed_count > 1) %>%
        nrow()

multiple_passed

[1] 0

In [32]:
correct_counts <- submissions_data %>%
  filter(submission_status == "correct") %>%
  group_by(user_id, step_id) %>%
  summarise(correct_count = n(), .groups = "drop")

multiple_correct <- correct_counts %>%
  filter(correct_count > 1) %>%
        nrow()
        
multiple_correct

[1] 5066

In [33]:
comparison <- events_data %>%
  filter(action == "passed") %>%
  select(user_id, step_id, first_passed_time = timestamp_dt) %>%
  full_join(
    submissions_data %>%
      filter(submission_status == "correct") %>%
      group_by(user_id, step_id) %>%
      summarise(first_correct_time = min(timestamp_dt), .groups = "drop"),
    by = c("user_id", "step_id")
  ) %>%
  mutate(passed_before_correct = first_passed_time < first_correct_time)


comparison %>% filter(passed_before_correct)

user_id,step_id,first_passed_time,first_correct_time,passed_before_correct
<dbl>,<dbl>,<dttm>,<dttm>,<lgl>


In [34]:
submissions_data %>%
    anti_join(
        events_data %>%
            filter(action == "started_attempt") %>%
            select(user_id, step_id),
        by = c("user_id", "step_id")
    ) %>%
        nrow()


[1] 6

In [35]:
events_data %>%
  arrange(user_id, timestamp_dt) %>%
  group_by(user_id) %>%
  filter(timestamp_dt < lag(timestamp_dt))



step_id,timestamp,action,user_id,timestamp_dt
<dbl>,<dbl>,<chr>,<dbl>,<dttm>


In [36]:
submissions_data %>%
  arrange(user_id, step_id, timestamp_dt) %>%
  group_by(user_id, step_id) %>%
  filter(timestamp_dt < lag(timestamp_dt))

step_id,timestamp,submission_status,user_id,timestamp_dt
<dbl>,<dbl>,<chr>,<dbl>,<dttm>


In [37]:
submissions_data %>%
  inner_join(events_data %>% 
               filter(action=="discovered") %>% 
               select(user_id, step_id, discovered_time = timestamp_dt),
             by=c("user_id","step_id")) %>%
  filter(timestamp_dt < discovered_time) %>%
    nrow()

[1] 203

## Preprocessing

In [38]:

user_level_data = data.frame(user_id = unique(events_data$user_id))

In [39]:
events_bounds <- events_data %>%
  group_by(user_id) %>%
  summarise(
    first_activity_events = min(timestamp_dt, na.rm = TRUE),
    last_activity_events  = max(timestamp_dt, na.rm = TRUE),
    .groups = "drop"
  )

submissions_bounds <- submissions_data %>%
  group_by(user_id) %>%
  summarise(
    first_activity_subm = min(timestamp_dt, na.rm = TRUE),
    last_activity_subm  = max(timestamp_dt, na.rm = TRUE),
    .groups = "drop"
  )

activity_bounds <- full_join(events_bounds, submissions_bounds, by = "user_id") %>%
  mutate(
    first_activity = pmin(first_activity_events, first_activity_subm, na.rm = TRUE),
    last_activity  = pmax(last_activity_events, last_activity_subm, na.rm = TRUE),
    activity_span_days = as.numeric(last_activity - first_activity, units = "days"),
    activity_span_days = ifelse(activity_span_days == 0, 1, activity_span_days)
  ) %>%
  select(user_id, first_activity, last_activity, activity_span_days)

user_level_data <- left_join(user_level_data, activity_bounds, by = "user_id")


In [40]:
activity_days <- bind_rows(
    events_data %>% select(user_id, timestamp_dt),
    submissions_data %>% select(user_id, timestamp_dt)
  ) %>%
  mutate(day = as_date(timestamp_dt)) %>%
  distinct(user_id, day)

active_days_df <- activity_days %>%
  count(user_id, name = "active_days")

median_gap_df <- activity_days %>%
  arrange(user_id, day) %>%
  group_by(user_id) %>%
  summarise(
    median_gap_days = if (n() >= 2) median(as.numeric(diff(day))) else 0,
    .groups = "drop"
  )

user_level_data <- user_level_data %>%
  left_join(active_days_df, by = "user_id") %>%
  left_join(median_gap_df, by = "user_id")

In [41]:
steps <- sort(unique(events_data$step_id))
step_score_vec <- rep(0, length(steps))

score <- list(
  `1` = c(3,14,15,18,19,22,30,31,32,33,38,39,40,47,49,53,54,55,56,57,
          62,64,65,68,73,74,76,80,81,82,83,95,100,104,108,109,111,
          122,123,131,133,138,139,140,141,147,148,149,155,156,157,
          161,162,163,169,170,172,177,185,186),

  `2` = c(41,63,94,99,113,114,115,125,132,179,189),

  `3` = c(126,180)
)

# Векторизация загрузки score
step_score_vec[unlist(score)] <- rep(names(score), lengths(score))

step_index <- match(events_data$step_id, steps)

events_data <- events_data %>%
  mutate(step_score = ifelse(action == "passed",
                             as.numeric(step_score_vec[step_index]), 0))

In [42]:
score_steps_df <- events_data %>%
  group_by(user_id) %>%
  summarise(
    total_score = sum(step_score),
    total_steps = n_distinct(step_id),
    .groups = "drop"
  )

user_level_data <- user_level_data %>%
  left_join(score_steps_df, by = "user_id")

In [43]:
user_level_data <- user_level_data %>%
  mutate(
    steps_per_day   = total_steps / active_days,
    score_per_day   = total_score / active_days,
    progress_percent = total_score / 88 * 100,
    steps_percent    = total_steps / 198 * 100,
    certificate      = ifelse(total_score >= 75, 1, 0),
    risk_time_threshold = last_activity + 2 * median_gap_days*24*3600
  )


In [44]:
end_observation <- max(
  max(events_data$timestamp_dt, na.rm = TRUE),
  max(submissions_data$timestamp_dt, na.rm = TRUE)
)

In [45]:
user_level_data <- user_level_data %>%
  mutate(
          drop_label = ifelse(
           certificate == 1 | risk_time_threshold > end_observation, 0,  
           1  
         ))

In [46]:
user_level_data %>%
    summarise(across(everything(),n_distinct)) %>%
        pivot_longer(
            cols = everything(),
            names_to = "column",
            values_to = "unique_values")
            

column,unique_values
<chr>,<int>
user_id,19234
first_activity,19223
last_activity,19226
activity_span_days,11581
active_days,121
median_gap_days,440
total_score,89
total_steps,198
steps_per_day,1993


## EDA on Aggregated User-Level Features

---

In [47]:
dim(user_level_data)

[1] 19234    15

In [48]:
glimpse(user_level_data)

Rows: 19,234
Columns: 15
$ user_id             <dbl> 17632, 12494, 442, 22254, 6646, 5681, 19739, 9244,…
$ first_activity      <dttm> 2015-06-15 04:00:48, 2015-06-15 04:18:51, 2015-06…
$ last_activity       <dttm> 2017-09-17 11:59:26, 2015-10-07 10:36:40, 2017-02…
$ activity_span_days  <dbl> 825.3323843, 114.2623727, 617.7395486, 26.9287500,…
$ active_days         <int> 20, 21, 45, 13, 14, 23, 13, 10, 22, 4, 20, 5, 32, …
$ median_gap_days     <dbl> 1.0, 2.0, 1.0, 1.0, 5.0, 2.0, 47.0, 2.0, 2.0, 2.0,…
$ total_score         <dbl> 34, 28, 36, 22, 83, 78, 29, 27, 31, 6, 88, 6, 81, …
$ total_steps         <int> 119, 79, 91, 60, 186, 168, 74, 61, 71, 11, 193, 12…
$ steps_per_day       <dbl> 5.950000, 3.761905, 2.022222, 4.615385, 13.285714,…
$ score_per_day       <dbl> 1.7000000, 1.3333333, 0.8000000, 1.6923077, 5.9285…
$ progress_percent    <dbl> 38.636364, 31.818182, 40.909091, 25.000000, 94.318…
$ steps_percent       <dbl> 60.1010101, 39.8989899, 45.9595960, 30.3030303, 93…
$ certificate  

In [49]:
summary(user_level_data)

    user_id      first_activity                  
 Min.   :    1   Min.   :2015-06-15 04:00:48.00  
 1st Qu.: 6636   1st Qu.:2015-11-14 21:38:04.50  
 Median :13358   Median :2016-10-25 18:24:33.00  
 Mean   :13364   Mean   :2016-11-06 22:51:33.26  
 3rd Qu.:20074   3rd Qu.:2017-10-15 10:58:04.00  
 Max.   :26798   Max.   :2018-05-19 23:32:30.00  
 last_activity                    activity_span_days   active_days    
 Min.   :2015-06-15 04:54:21.00   Min.   :   0.0000   Min.   :  1.00  
 1st Qu.:2016-02-29 12:23:03.75   1st Qu.:   0.3046   1st Qu.:  1.00  
 Median :2017-03-03 09:44:51.00   Median :   1.0000   Median :  1.00  
 Mean   :2017-01-18 18:07:26.49   Mean   :  73.0520   Mean   :  6.09  
 3rd Qu.:2017-12-16 22:03:24.50   3rd Qu.:  38.4036   3rd Qu.:  6.00  
 Max.   :2018-05-19 23:33:31.00   Max.   :1064.7378   Max.   :478.00  
 median_gap_days     total_score     total_steps     steps_per_day      
 Min.   :   0.000   Min.   : 0.00   Min.   :  1.00   Min.   :  0.01176  
 1st Qu

In [50]:
summary(user_level_data$activity_span_days)

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
   0.0000    0.3046    1.0000   73.0520   38.4036 1064.7378 

In [51]:
colSums(is.na(user_level_data))

user_id      first_activity       last_activity  activity_span_days 
                  0                   0                   0                   0 
        active_days     median_gap_days         total_score         total_steps 
                  0                   0                   0                   0 
      steps_per_day       score_per_day    progress_percent       steps_percent 
                  0                   0                   0                   0 
        certificate risk_time_threshold          drop_label 
                  0                   0                   0

In [52]:
sum(duplicated(user_level_data))


[1] 0

In [53]:
sum(user_level_data$first_activity > user_level_data$last_activity)
sum(user_level_data$active_days > user_level_data$activity_span_days)

[1] 0

[1] 6178